# F1 Aerodynamics GEM Training — Kaggle

Trains `F1AeroNet` (Gauge-Equivariant Mesh CNN) to predict surface pressure (Cp),
wall shear stress (WSS), and drag coefficient (Cd) on F1 car geometries.

**Cells in order:**
1. GPU / environment check
2. Install dependencies
3. Clone code from GitHub
4. Copy dataset from Kaggle input
5. Write Kaggle training config
6. Pre-compute Cd normalisation stats
7. Run training
8. Save checkpoints to `/kaggle/working/output`

## 1 — GPU / Environment Check

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("No GPU detected — training will be slow on CPU.")

print(f"PyTorch: {torch.__version__}")

# Capture version strings for dependency install below
TORCH_VER = torch.__version__.split('+')[0]   # e.g. "2.3.0"
CUDA_TAG  = f"cu{torch.version.cuda.replace('.', '')}" if torch.cuda.is_available() else "cpu"
print(f"\nInstall tag: torch-{TORCH_VER}+{CUDA_TAG}")

## 2 — Install Dependencies

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

# torch-geometric (core, no C extensions needed for basic use)
pip('torch-geometric')

# Optional compiled extensions — provide significant speedup on CUDA
pyg_url = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'
try:
    pip('torch-scatter', 'torch-sparse', '-f', pyg_url)
    print('torch-scatter / torch-sparse installed.')
except Exception as e:
    print(f'[WARN] Could not install C extensions ({e}). Falling back to pure-Python ops.')

# CFD mesh I/O and scientific stack
pip('pyvista', 'vtk', 'scipy', 'pyyaml', 'tqdm')

print('\nAll dependencies installed.')

## 3 — Clone Code from GitHub

In [ ]:
import os, subprocess

REPO_URL  = 'https://github.com/aumrawal/GDL_Cars.git'
REPO_DIR  = '/kaggle/working/GDL_Cars'

if not os.path.exists(REPO_DIR):
    print(f'Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Clone complete.')
else:
    print(f'{REPO_DIR} already exists — pulling latest changes.')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# Add repo to Python path so imports work
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('\nRepo contents:')
for f in sorted(os.listdir(REPO_DIR)):
    print(f'  {f}')

## 4 — Copy Dataset

In [ ]:
import shutil, os

SRC_DATASET = '/kaggle/input/datasets/aumrawal29/drivaernet-200'
DST_DATASET = '/kaggle/working/drivaernet_real'

if not os.path.exists(DST_DATASET):
    print(f'Copying dataset...\n  {SRC_DATASET}\n  → {DST_DATASET}')
    shutil.copytree(SRC_DATASET, DST_DATASET)
    print('Copy complete.')
else:
    print(f'{DST_DATASET} already exists, skipping copy.')

# Verify structure
print('\nDataset top-level:')
for entry in sorted(os.listdir(DST_DATASET)):
    full = os.path.join(DST_DATASET, entry)
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f'  {entry}/  ({count} items)')
    else:
        print(f'  {entry}')

mesh_dir = os.path.join(DST_DATASET, 'meshes')
if os.path.isdir(mesh_dir):
    vtps = [f for f in os.listdir(mesh_dir) if f.endswith('.vtp')]
    print(f'\nFound {len(vtps)} VTP meshes in {mesh_dir}')
else:
    print(f'[WARNING] No meshes/ directory found in {DST_DATASET}.')
    print('Expected: drivaernet_real/meshes/<design_id>.vtp')

## 5 — Write Kaggle Training Config

Overrides the base config with Kaggle-specific paths and **the 3 Cd-collapse fixes**:
- `cd` loss weight → **1.0** (equal weight with Cp/WSS — correct after y_cd z-score normalisation brings all loss scales to the same order)
- `cd` loss type → **mse** (was l1 — MSE gradient grows with error magnitude, fighting mean collapse)
- Cd z-score normalisation applied in dataset via `cd_stats.json` (handled in code)

In [ ]:
import yaml, os

CHECKPOINT_DIR = '/kaggle/working/GDL_Cars/runs/'

kaggle_cfg = {
    'data': {
        'data_root':    '/kaggle/working/drivaernet_real',
        'max_vertices': 50000,
        'rho':          1.225,
        'U_inf':        83.33,
        'L_ref':        5.0,
    },
    'model': {
        'in_channels':           9,
        # (24,2) removed — its K_neigh tensor (E×120×80×fp32) caused OOM on T4.
        # (16,2) keeps K_neigh at E×80×80×fp32 (1.15 GB at E=48k vs 1.72 GB before).
        'layer_types':           [[16, 2], [16, 2], [16, 2], [24, 1], [16, 1]],
        'nonlin_samples':        5,
        'head_dropout':          0.1,
        'break_symmetry_final':  True,
    },
    'training': {
        'epochs':         200,
        'batch_size':     1,
        'lr':             1e-3,
        'weight_decay':   0.0,
        'T_0':            50,
        'T_mult':         2,
        'min_lr':         1e-6,
        'grad_clip':      1.0,
        'save_every':     50,
        'checkpoint_dir': CHECKPOINT_DIR,
        'log_every':      10,
        'loss_weights': {
            'cp':  1.0,
            'wss': 1.0,
            'cd':  1.0,
            'cl':  0.0,
        },
        'loss_type': {
            'cp':  'huber',
            'wss': 'huber',
            'cd':  'mse',
            'cl':  'l1',
        },
    },
    'eval': {
        'checkpoint': os.path.join(CHECKPOINT_DIR, 'best.pt'),
        'output_dir': '/kaggle/working/output',
        'save_vtk':   True,
    },
}

cfg_path = os.path.join(REPO_DIR, 'configs', 'f1_kaggle.yaml')
with open(cfg_path, 'w') as f:
    yaml.dump(kaggle_cfg, f, default_flow_style=False, sort_keys=False)

print(f'Config written to {cfg_path}')
print(yaml.dump(kaggle_cfg, default_flow_style=False, sort_keys=False))

## 6 — Pre-compute Cd Normalisation Stats

Checks for a previously saved `cd_stats.json` in Kaggle input datasets first —
attach the output from a prior run as an input dataset to skip recomputation entirely.
If not found, builds the training cache from VTPs and computes stats from scratch.

Also clears stale `.pt` cache files (CACHE_VERSION 1 → 2) before processing.

In [ ]:
import os, json, glob, shutil, torch, sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

DATA_ROOT  = '/kaggle/working/drivaernet_real'
STATS_PATH = os.path.join(DATA_ROOT, 'cd_stats.json')

MIN_SAMPLES = 50   # reject stats computed from fewer than this many samples

# Fallback stats computed from the full 210-sample training split.
# Used only if no valid cd_stats.json is found or recomputed.
FALLBACK_STATS = {
    'cd_mean':   0.241828,
    'cd_std':    0.017059,
    'n_samples': 210,
}

def _load_and_validate_stats(path):
    """Return stats dict if valid (n >= MIN_SAMPLES and std > 0.01), else None."""
    try:
        with open(path) as f:
            s = json.load(f)
        n   = s.get('n_samples', 0)
        std = s.get('cd_std', 0)
        if n < MIN_SAMPLES:
            print(f'[WARN] cd_stats.json has only {n} samples (need >= {MIN_SAMPLES}) — discarding.')
            return None
        if std < 0.01:
            print(f'[WARN] cd_stats.json std={std:.6f} is suspiciously small — discarding.')
            return None
        return s
    except Exception as e:
        print(f'[WARN] Could not read cd_stats.json: {e}')
        return None

# ── Restore cd_stats.json from a prior run's output dataset ─────────────────
if os.path.exists(STATS_PATH):
    stats = _load_and_validate_stats(STATS_PATH)
    if stats is None:
        print('Removing invalid local cd_stats.json.')
        os.remove(STATS_PATH)

if not os.path.exists(STATS_PATH):
    for candidate in glob.glob('/kaggle/input/**/cd_stats.json', recursive=True):
        stats = _load_and_validate_stats(candidate)
        if stats is not None:
            shutil.copy2(candidate, STATS_PATH)
            print(f'Restored valid cd_stats.json from prior run: {candidate}')
            print(f"  n={stats['n_samples']}  mean={stats['cd_mean']:.6f}  std={stats['cd_std']:.6f}")
            break
        else:
            print(f'Skipping invalid candidate: {candidate}')

# ── Stale cache clearing ─────────────────────────────────────────────────────
processed_dir = os.path.join(DATA_ROOT, 'processed')
if os.path.exists(processed_dir):
    stale_count = 0
    for split in os.listdir(processed_dir):
        split_dir = os.path.join(processed_dir, split)
        if os.path.isdir(split_dir):
            for fname in os.listdir(split_dir):
                if fname.endswith('.pt'):
                    pt_path = os.path.join(split_dir, fname)
                    try:
                        d = torch.load(pt_path, weights_only=False)
                        if getattr(d, 'cache_version', torch.tensor([0])).item() < 2:
                            os.remove(pt_path)
                            stale_count += 1
                    except Exception:
                        os.remove(pt_path)
                        stale_count += 1
    if stale_count:
        print(f'Cleared {stale_count} stale .pt cache files (CACHE_VERSION < 2).')
    else:
        print('No stale cache files found.')

# ── Cd normalisation stats ───────────────────────────────────────────────────
from data.drivaernet_dataset import DrivAerNetDataset

if os.path.exists(STATS_PATH):
    with open(STATS_PATH) as f:
        s = json.load(f)
    print(f'\nCd stats ready (from {s.get("n_samples", "?")} samples):')
    print(f"  mean = {s['cd_mean']:.6f}")
    print(f"  std  = {s['cd_std']:.6f}")
else:
    print('Computing Cd stats from full training split...')
    print('(Converts VTP → .pt — takes a few minutes on first run.)\n')

    train_ds = DrivAerNetDataset(data_root=DATA_ROOT, split='train', normalize_cd=False)
    cd_vals  = []
    for i in range(len(train_ds)):
        data = train_ds[i]
        if hasattr(data, 'y_cd') and data.y_cd is not None:
            val = float(data.y_cd.reshape(-1)[0])
            if val != 0.0:
                cd_vals.append(val)
        if (i + 1) % 20 == 0 or (i + 1) == len(train_ds):
            print(f'  processed {i+1}/{len(train_ds)}  (Cd values found so far: {len(cd_vals)})')

    print(f'\nTotal Cd values collected: {len(cd_vals)}')
    if cd_vals:
        print(f'  range : [{min(cd_vals):.4f}, {max(cd_vals):.4f}]')

    if len(cd_vals) >= MIN_SAMPLES:
        t = torch.tensor(cd_vals, dtype=torch.float32)
        s = {
            'cd_mean':   float(t.mean()),
            'cd_std':    float(t.std()),
            'n_samples': len(cd_vals),
        }
    else:
        print(f'[WARN] Only {len(cd_vals)} Cd values found — using hardcoded fallback stats.')
        s = FALLBACK_STATS

    with open(STATS_PATH, 'w') as f:
        json.dump(s, f, indent=2)
    print(f'\nCd stats saved → mean={s["cd_mean"]:.6f}  std={s["cd_std"]:.6f}'
          f'  n={s["n_samples"]}'
          + (' (FALLBACK)' if s is FALLBACK_STATS else ''))

## 7 — Run Training

In [ ]:
import os, sys, time, csv, shutil
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# Reduce memory fragmentation — allocator can reuse non-contiguous segments.
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import yaml
from train.trainer import load_datasets, save_checkpoint, load_checkpoint
from train.losses import F1AeroLoss
from models.f1_net import F1AeroNet

cfg_path = os.path.join(REPO_DIR, 'configs', 'f1_kaggle.yaml')
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

train_cfg  = cfg['training']
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_dir   = train_cfg['checkpoint_dir']
output_dir = cfg['eval']['output_dir']
os.makedirs(ckpt_dir,   exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# ── Data, model, optimiser ───────────────────────────────────────────────────
train_loader, val_loader = load_datasets(cfg)

model = F1AeroNet.from_config(cfg['model']).to(device)
param_counts = model.count_parameters()
print('Model parameters:')
for k, v in param_counts.items():
    print(f'  {k:20s}: {v:>10,}')

optimizer = AdamW(model.parameters(), lr=train_cfg['lr'],
                  weight_decay=train_cfg['weight_decay'])
scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0=train_cfg.get('T_0', 50),
    T_mult=train_cfg.get('T_mult', 2),
    eta_min=train_cfg['min_lr'],
)
criterion = F1AeroLoss(
    weights=train_cfg['loss_weights'],
    loss_types=train_cfg['loss_type'],
).to(device)

# ── Resume ───────────────────────────────────────────────────────────────────
start_epoch = 0
best_val    = float('inf')

# Try resuming from output dir first (survives Kaggle session loss),
# then ckpt_dir, then runs_new/ committed to the repo.
for resume_path in [os.path.join(output_dir, 'last.pt'),
                    os.path.join(ckpt_dir,   'last.pt'),
                    os.path.join(REPO_DIR,   'runs_new', 'last.pt')]:
    if os.path.exists(resume_path):
        start_epoch, best_val = load_checkpoint(resume_path, model, optimizer, scheduler)
        print(f'Resumed from {resume_path}  epoch={start_epoch}  best_val={best_val:.5f}')
        break

# ── CSV log ──────────────────────────────────────────────────────────────────
log_path   = os.path.join(ckpt_dir, 'training_log.csv')
log_file   = open(log_path, 'a', newline='')
log_writer = csv.DictWriter(log_file,
    fieldnames=['epoch', 'lr', 'train_total', 'train_cp', 'train_wss',
                'train_cd', 'train_cl', 'val_total', 'val_cp', 'val_wss',
                'val_cd', 'val_cl', 'time_s'])
if start_epoch == 0:
    log_writer.writeheader()

# ── OOM safety ───────────────────────────────────────────────────────────────
MAX_EDGES  = 150_000
save_every = train_cfg.get('save_every', 50)
grad_clip  = train_cfg['grad_clip']
log_every  = train_cfg['log_every']

print(f'\nTraining for {train_cfg["epochs"]} epochs on {device}')
print(f'MAX_EDGES={MAX_EDGES}  |  periodic save every {save_every} epochs\n')

# ── Helper: one epoch ────────────────────────────────────────────────────────
def run_epoch(loader, train=True):
    model.train(train)
    total, parts, n = 0.0, {}, 0
    for i, batch in enumerate(loader):
        if batch.edge_index.shape[1] > MAX_EDGES:
            print(f'  [SKIP] {getattr(batch,"design_id","?")} '
                  f'({batch.edge_index.shape[1]} edges > {MAX_EDGES})')
            continue
        batch = batch.to(device)
        with torch.set_grad_enabled(train):
            pred = model(
                x=batch.x, edge_index=batch.edge_index,
                angles=batch.edge_angles, transporters=batch.edge_transporters,
                batch=batch.batch if hasattr(batch, 'batch') else None,
            )
            loss, prt = criterion(pred, batch)

        if torch.isnan(loss):
            print(f'  [NaN loss at batch {i}]')
            continue

        if train:
            optimizer.zero_grad()
            loss.backward()
            if grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            torch.cuda.empty_cache()

        total += loss.item()
        for k, v in prt.items():
            parts[k] = parts.get(k, 0.0) + v.item()
        n += 1

        if train and (i + 1) % log_every == 0:
            print(f'    batch {i+1:4d}/{len(loader)}  '
                  f'loss={total/n:.5f}  '
                  + '  '.join(f'{k}={parts[k]/n:.5f}' for k in parts if parts[k] > 0))

    n = max(n, 1)
    return {'total': total/n, **{k: v/n for k, v in parts.items()}}

# ── Epoch loop ───────────────────────────────────────────────────────────────
for epoch in range(start_epoch + 1, train_cfg['epochs'] + 1):
    t0 = time.time()
    print(f"Epoch {epoch:3d}/{train_cfg['epochs']}  lr={optimizer.param_groups[0]['lr']:.2e}")

    train_losses = run_epoch(train_loader, train=True)
    val_losses   = run_epoch(val_loader,   train=False)
    elapsed = time.time() - t0
    scheduler.step()

    print(f"  TRAIN  total={train_losses['total']:.5f}  "
          + '  '.join(f"{k}={train_losses.get(k,0):.5f}" for k in ['cp','wss','cd','cl']))
    print(f"  VAL    total={val_losses['total']:.5f}  "
          + '  '.join(f"{k}={val_losses.get(k,0):.5f}" for k in ['cp','wss','cd','cl'])
          + f'  [{elapsed:.1f}s]')

    log_writer.writerow({
        'epoch': epoch, 'lr': optimizer.param_groups[0]['lr'],
        'train_total': train_losses['total'],
        'train_cp': train_losses.get('cp', 0), 'train_wss': train_losses.get('wss', 0),
        'train_cd': train_losses.get('cd', 0), 'train_cl': train_losses.get('cl', 0),
        'val_total': val_losses['total'],
        'val_cp': val_losses.get('cp', 0), 'val_wss': val_losses.get('wss', 0),
        'val_cd': val_losses.get('cd', 0), 'val_cl': val_losses.get('cl', 0),
        'time_s': elapsed,
    })
    log_file.flush()

    # ── Save last.pt and sync to output dir every epoch ──────────────────────
    last_ckpt = os.path.join(ckpt_dir, 'last.pt')
    save_checkpoint(last_ckpt, model, optimizer, scheduler, epoch, best_val, cfg)
    shutil.copy2(last_ckpt,  os.path.join(output_dir, 'last.pt'))
    shutil.copy2(log_path,   os.path.join(output_dir, 'training_log.csv'))

    # ── Save best.pt if val improved ─────────────────────────────────────────
    if val_losses['total'] < best_val:
        best_val = val_losses['total']
        best_ckpt = os.path.join(ckpt_dir, 'best.pt')
        save_checkpoint(best_ckpt, model, optimizer, scheduler, epoch, best_val, cfg)
        shutil.copy2(best_ckpt, os.path.join(output_dir, 'best.pt'))
        print(f'  ✓ New best model saved (val={best_val:.5f})')

    # ── Periodic checkpoint every save_every epochs ──────────────────────────
    if epoch % save_every == 0:
        periodic_name = f'checkpoint_epoch_{epoch:04d}.pt'
        periodic_path = os.path.join(ckpt_dir, periodic_name)
        save_checkpoint(periodic_path, model, optimizer, scheduler, epoch, best_val, cfg)
        shutil.copy2(periodic_path, os.path.join(output_dir, periodic_name))
        print(f'  ✓ Periodic checkpoint: {periodic_name}')

log_file.close()
print(f'\nTraining complete. Best val loss: {best_val:.5f}')
print(f'Best checkpoint : {os.path.join(output_dir, "best.pt")}')

## 8 — Save Checkpoints to Output

In [ ]:
import shutil, os

CHECKPOINT_DIR = '/kaggle/working/GDL_Cars/runs/'
OUTPUT_DIR     = '/kaggle/working/output'

os.makedirs(OUTPUT_DIR, exist_ok=True)

for fname in ['best.pt', 'last.pt', 'training_log.csv']:
    src = os.path.join(CHECKPOINT_DIR, fname)
    dst = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'Saved: {dst}  ({size_mb:.1f} MB)')
    else:
        print(f'[NOT FOUND] {src}')

# Also copy Cd stats so they can be reused in evaluation
stats_src = '/kaggle/working/drivaernet_real/cd_stats.json'
if os.path.exists(stats_src):
    shutil.copy2(stats_src, os.path.join(OUTPUT_DIR, 'cd_stats.json'))
    print(f'Saved: {OUTPUT_DIR}/cd_stats.json')

print(f'\nOutput directory contents:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

## (Optional) Cd Scatter Plot — Sanity Check

Re-run after training to verify Cd predictions have spread away from the mean.

> **Note:** after Fix 2, `y_cd` and `pred['cd']` are both in **normalised** space
> (z-scored, mean≈0, std≈1). To convert back to physical Cd, load `cd_stats.json`
> and apply `cd_physical = cd_norm * cd_std + cd_mean`.

In [ ]:
import os, glob, json, torch
import matplotlib.pyplot as plt
import numpy as np

# Load Cd stats for denormalisation
with open('/kaggle/working/drivaernet_real/cd_stats.json') as f:
    cd_stats = json.load(f)
CD_MEAN = cd_stats['cd_mean']
CD_STD  = cd_stats['cd_std']

# Load best checkpoint
CKPT   = '/kaggle/working/output/best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from models.f1_net import F1AeroNet
from data.drivaernet_dataset import DrivAerNetDataset

ckpt      = torch.load(CKPT, map_location='cpu', weights_only=False)
saved_cfg = ckpt.get('cfg', {})
model     = F1AeroNet.from_config(saved_cfg.get('model', {}))
model.load_state_dict(ckpt['model'])
model     = model.to(device).eval()

# Collect predictions
val_ds = DrivAerNetDataset(
    data_root='/kaggle/working/drivaernet_real', split='val'
)
val_ds.set_cd_stats(CD_MEAN, CD_STD)

results = {'train': [], 'val': []}
for split in ['train', 'val']:
    ds = DrivAerNetDataset(
        data_root='/kaggle/working/drivaernet_real', split=split
    )
    ds.set_cd_stats(CD_MEAN, CD_STD)
    with torch.no_grad():
        for i in range(len(ds)):
            data = ds[i].to(device)
            pred = model(
                x=data.x, edge_index=data.edge_index,
                angles=data.edge_angles, transporters=data.edge_transporters,
                batch=torch.zeros(data.x.shape[0], dtype=torch.long, device=device),
            )
            # Denormalise to physical Cd
            cd_pred = pred['cd'].cpu().item() * CD_STD + CD_MEAN
            cd_true = data.y_cd.cpu().item() * CD_STD + CD_MEAN
            results[split].append((cd_true, cd_pred))

# Plot
fig, ax = plt.subplots(figsize=(6, 6))
colours = {'train': '#4C72B0', 'val': '#DD8452'}
for split, pts in results.items():
    trues  = [p[0] for p in pts]
    preds  = [p[1] for p in pts]
    mae    = np.mean(np.abs(np.array(trues) - np.array(preds)))
    corr   = float(np.corrcoef(trues, preds)[0, 1]) if len(trues) > 2 else float('nan')
    ax.scatter(trues, preds, s=20, alpha=0.7, color=colours[split],
               label=f'{split} (n={len(pts)}, MAE={mae:.4f}, r={corr:.3f})')

all_true = [p[0] for pts in results.values() for p in pts]
lo, hi   = min(all_true) * 0.98, max(all_true) * 1.02
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='Perfect')
ax.set_xlabel('True Cd (physical)')
ax.set_ylabel('Predicted Cd (physical)')
ax.set_title(f"Cd Prediction (epoch={ckpt.get('epoch', '?')})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/kaggle/working/output/cd_scatter.png', dpi=150)
plt.show()
print('Scatter plot saved.')